# moveBoxes · 지금까지 가장 좋았던 성능과 학습 루트

**로컬에 보관된 실제 완료 평가 기록 기준 · 정리일 2026-09-16 · State Stage ACT**

한 파일에서 **성과와 학습 계보 확인 → 과거 최고 가중치 복원 → 재평가/영상 → 당시 학습 설정으로 재학습 → 패키징**까지 실행합니다.
이 문서의 과거 점수는 저장된 기록이며, 이 노트북으로 새 학습·GPU 평가를 실행한 점수가 아닙니다.
외부에서 새로 진행된 실험 중 이 작업 폴더에 기록이 없는 결과는 포함하지 않았습니다.

| 난이도 | 확인된 최고 분류 정확도 | 실제 정답 상자 / 전체 | 평가 조건 | 보존할 모델 |
|---|---:|---:|---|---|
| Easy | **100%** | 200 / 200 | 최종 100회, seed 70000–70099 | easy_lab_v1 / block_03.pt, 추가 6,000 update |
| Medium | **40.625%** | 13 / 32 | 테스트 8회, seed 40000–40007 | medium_lab_v1 / block_02.pt, 추가 4,000 update |
| Hard | **4.1667%** | 2 / 48 | 테스트 8회, seed 48000–48007 | hard_stage_v1 / best_val.pt |

**SORT ACCURACY는 정답 분류한 상자의 비율입니다. 에피소드 완주 성공률과 다릅니다.**
모두 episode 제한 200, 기록된 rollout은 199행동입니다. Easy는 고정 배치라 seed 100개의 성공이 100개 서로 다른 배치의 일반화를 뜻하지 않습니다.
Medium·Hard는 표본 8회 결과이므로 Easy의 최종 100회 결과와 같은 확실성으로 해석하지 않습니다.
서로 다른 시드/평가 단계의 이 세 숫자를 합쳐 공식 종합 점수라고 보고하지 않습니다.

읽기만 할 때는 실행할 필요가 없습니다. 실행은 **Colab GPU(T4) 또는 동등한 Linux GPU 환경**을 전제로 합니다.
기존 GitHub 인증 → 설치 → Release 데이터 다운로드 흐름을 유지하며, Drive와 별도 Python 파일 업로드는 필요 없습니다.
코드·데이터·체크포인트 다운로드에는 인터넷, 결과 백업에는 GitHub 토큰이 필요합니다.

## 1. 성능을 만든 학습 경로

### Easy: Stage ACT 기존 가중치 → 첫 집기 보강 → 6,000 update 모델 보존

`state 원본 200개 + 성공/복구 시연 16개 → stage_act_v1의 Easy 가중치 → easy_lab_v1 → block_03.pt`

- 무작위 초기화 학습이 아니라 **기존 Easy 가중치를 초기값으로 한 추가 학습**입니다. optimizer는 새로 시작했습니다.
- 첫 집기 구간을 30% 확률로 별도 표집했습니다. 나머지는 전체 작업·복구·단계 전환을 학습하므로 이후 상자의 집기도 포함됩니다.
- 2,000 update마다 개발 8회 평가. 개발 최고 정확도는 2,000회 **68.75%** → 4,000회 **0%** → 6,000회 **100%** → 8,000회 **100%**였습니다.
- 처음 100%에 도달한 `block_03.pt`를 보존했고, 별도 테스트 8회와 최종 100회에서도 100%를 기록했습니다.
- 총 scheduler 예산은 20,000입니다. **6,000에서 중단하더라도 total_iters를 6,000으로 바꾸면 학습률 곡선이 달라집니다.**

### Medium: Medium 자체 기존 가중치 → 같은 first-pick 학습 → 4,000 update 모델 보존

`state 원본 200개 + 성공/복구 시연 16개 → stage_act_v1의 Medium 가중치 → medium_lab_v1 → block_02.pt`

- Easy 모델을 Medium으로 전이해서 얻은 기록이 아닙니다. 72차원 Medium 모델을 별도로 이어 학습했습니다.
- Easy와 같은 prior 행동 학습과 first-pick 30%, 2,000 update 단위 평가를 사용했습니다.
- 개발 seed 50000–50007에서 `block_02`는 **15.625%**, 별도 테스트 seed 40000–40007에서 **40.625%**였습니다.
- 6,000/8,000/10,000/12,000 update의 개발 최고값은 3.125/12.5/12.5/9.375%였습니다. 더 오래 학습했다고 개선되지 않았습니다.
- 이후 v2.2에서 **같은 SHA-256 모델**을 다른 시드로 평가한 기록은 28.125%입니다. 모델 개선 수치로 취급하지 않습니다.

### Hard: Hard 원본 + 24개 복구 시연 → 독립 Stage ACT 학습 → best_val 보존

`state 원본 200개 + 6상자 성공/복구 시연 24개 → Hard Stage ACT 30,000 update → best_val.pt`

- 저장된 snapshot에는 초기 가중치 파일이 없습니다. 아래 재학습 셀은 실제 저장된 training job의 warm_start 유무를 검증합니다.
- 완료 학습은 30,000 update입니다. validation loss 최저 기록은 18,000 update에 있으며, 복원 셀에서 실제 best_val.pt의 step도 출력합니다.
- 8회 테스트에서 정답 상자는 총 2개입니다. **현재까지 확인된 Hard 기준 모델이며 실용적인 성공 모델로 보기는 어렵습니다.**
- 확인한 Hard target_v2와 Easy/Medium transfer_v3의 테스트는 모두 0%였습니다. 이들을 최고 모델로 대체하지 않습니다.

위 경로는 학습 계보입니다. 최고 checkpoint를 그대로 쓰는 **복원**과 초기 가중치부터 다시 도는 **재학습**은 별도 폴더에 저장합니다.
새로운 all-pick, DAgger, PPO/RL 경로는 여기서 확인한 최고 기록의 생성 경로가 아니므로 기본 학습에 섞지 않습니다.

## 2. 보존해야 할 설정과 재현 범위

| 항목 | 기록된 설정 |
|---|---|
| 관측 / 출력 | State 54 / 72 / 90차원, action XYZ + gripper 4차원 |
| 모델 | history 16, chunk 16, width 128, heads 4, layers 2, latent 16 |
| Easy·Medium 학습 | batch 64, lr 1e-4, seed 42, AMP, warmup 100, first_pick_fraction 0.30 |
| 행동 학습 | prior / zero-latent 경로 직접 지도학습 |
| 정책 실행 | 매 step 재추론, act_horizon 1, XYZ temporal ensemble 4, decay 0.25 |
| 단계 판단 | gate threshold 0.65, stage threshold 0.60 |
| 그리퍼 | 최신 예측 적용 |
| 데이터 | 원본 state HDF5 + 해당 run의 collection; 학습/검증 episode 분리 |
| 환경 소스 | 6048f33217f26ae39009a812f53c81171517f393 |

Hard의 세부 optimizer 설정은 추측하지 않고 저장된 `stage_train_job.json`을 다운로드해 표시하고 그대로 사용합니다.
Easy·Medium도 실제 job의 설정을 사용하며, 초기 가중치는 당시 snapshot에 있던 정확한 파일로 고정합니다.
이 노트북은 당시 모델/입력 파일을 SHA-256으로 고정하고 실행 코드는 명시된 Git commit으로 고정합니다.
학습 코드가 당시 기록과 다르면 그 차이를 `replay_origin.json`에 저장하고 출력합니다. 같은 점수나 byte 단위 가중치 재생성을 보장하지 않습니다.

**실행 순서**

1. 01~05: 설정, 코드, 인증, 설치, 데이터/GPU 확인.
2. 06~07: 검증된 과거 모델 복원, 당시 학습 job과 실제 checkpoint step 확인.
3. 08~10: 난이도별 공식 eval.py 재평가와 영상. 기본값은 과거와 같은 seed 목록입니다.
4. 11~14: `TRAIN_LEVELS`에 넣은 난이도만 당시 입력에서 재학습하고 평가. 기본값은 빈 목록입니다.
5. 15: 난이도별 과거 최고 가중치를 하나의 제출 ZIP으로 묶습니다. 재학습 결과는 별도 ZIP으로 저장합니다.

설정이 바뀌면 새 run_name을 사용합니다. 같은 설정으로 중단 후 재개할 때는 동일 run_name으로 01~07 실행 후 중단된 셀을 실행합니다.
기존 과거 Release와 checkpoint를 덮어쓰지 않습니다. 실행 중 결과 백업은 새 run의 Release에 저장됩니다.

In [ ]:
# 01 · 단계 ACT CONFIG · 기존 ACT/DP와 별도 결과
CFG = {
    # 저장 · ver2는 기존 DP 결과와 별도 Release에 저장
    'run_name': 'moveboxes_best_routes_review_v1',
    'profile': 'benchmark',
    'github_repository': 'SongYunu/moveBoxes',
    'project_dir': '/content/moveBoxes_best_routes',
    'project_ref': 'db12b8cc0bee547b9d2d3452abbc62187caab3d1',
    'output_root': '/content/moveboxes_runs',

    # 데이터 / Colab 2026.07 · Python 3.12 · T4
    'repo_dir': '/content/marso_best_routes_simulator',
    'repo_url': 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    'repo_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'data_dir': '/content/marso_data',
    'data_source': '/content/moveboxes_data_cache/marso_state_data.zip',
    'download_cache': '/content/moveboxes_data_cache',
    'packages': ['mani-skill==3.0.1', 'sapien==3.0.3', 'diffusers==0.38.0', 'hydra-core', 'omegaconf', 'gymnasium', 'tyro', 'h5py', 'kagglehub', 'tensorboard', 'matplotlib', 'transforms3d', 'imageio[ffmpeg]'],

    # 학습 · 시뮬레이터 없이 CPU 데이터 + GPU 모델
    'seed': 42,
    'num_demos': None,
    'batch_size': 64,
    'lr': 0.0001,
    'total_iters': {'easy': 12000, 'medium': 20000, 'hard': 30000},
    'amp': True,

    # 작은 State ACT · 기존 DP 체크포인트 사용 불가
    'history': 16,
    'chunk_size': 16,
    'width': 128,
    'heads': 4,
    'layers': 2,
    'latent_dim': 16,

    # 검증 / 중단 복구 / 작은 관측 위치 증강
    'save_freq': 2000,
    'warmup_steps': 500,
    'validation_batches': 8,
    'kl_weight': 0.001,
    'position_noise': 0.001,

    # 실행 · 매 스텝 재계획, 최근 XYZ 예측 평균, 집게는 최신 예측
    'temporal_decay': 0.25,
    'ensemble_window': 4,
    'ensemble_candidates': [1, 4],

    # 빠른 테스트 / 최종 평가 · 시드 분리, 기존 200스텝 유지
    'test_episodes': 8,
    'test_seed_start': 40000,
    'test_record_video': True,
    'tuning_episodes': 8,
    'tuning_seed_start': 20000,
    'benchmark_episodes': 100,
    'eval_seed_start': 30000,
    'max_episode_steps': {'easy': 200, 'medium': 200, 'hard': 200},
    'record_eval_video': True,

    # 출력
    'console_interval_seconds': 30,
    'team': 'my-team',

    # 실행 조건으로 행동 학습 · 빠른 테스트가 0이면 긴 평가 생략
    'action_training_mode': 'prior',
    'repair_iters': 2000,
    'allow_zero_success_evaluation': False,

    # 단계 판단 · 학습된 완료/복구 확신이 낮으면 현재 단계 유지
    'gate_threshold': 0.65,
    'stage_threshold': 0.6,
    'stage_loss_weight': 0.3,
    'gate_loss_weight': 0.3,

    # 복구 시연 · 수집 전용 expert, 학습/제출은 학습된 정책
    'recovery_episodes': 16,
    'recovery_max_attempts': 48,
    'recovery_seed_start': 100000,
    'collection_max_steps': {'easy': 500, 'medium': 900, 'hard': 1400},
    'noise_probability': 0.08,
    'action_noise_std': 0.12,
    'drop_probability': 0.015,

}

# 이 노트북의 실행 선택
TRAIN_LEVELS = []  # 재학습하려면 ['easy', 'medium', 'hard'] 중 원하는 난이도 지정
EVAL_PROTOCOL = 'historical'  # 'historical' 또는 'official_default' (4회)
# historical: Easy100회 / Medium8회 / Hard8회. official_default와 점수를 섞지 않습니다.


In [ ]:
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'colab_layout', 'build_modular_notebook',
             'colab_train_test_layout', 'build_train_test_notebook', 'colab_next_pick_layout',
             'build_next_pick_notebook', 'build_github_notebook', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'))
for name in ('act_v2_model','act_v2_data','act_v2_policy','act_v2_eval','act_v2_experiment','build_act_v2_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'/'stages'))
for name in ('stage_schema','stage_model','stage_policy','stage_labels','stage_data','stage_teacher',
             'stage_collect','stage_eval','stage_experiment','stage_chunk_policy',
             'stage_pick_sampling','stage_pick_train','stage_all_pick_retrain',
             'stage_pick_finetune','stage_pick_diagnose','stage_reference_check',
             'stage_anchor_continue','build_stage_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from stage_experiment import StageExperiment, source_bundle
experiment = StageExperiment(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])
print('코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.')


In [ ]:
# 03 · GitHub 인증 / 저장된 결과 복원
# 기존 환경변수 GH_TOKEN → Colab 보안 비밀 → 입력창 순서로 인증합니다.
# Fine-grained token: SongYunu/moveBoxes → Contents: Read and write.
# 이 저장소는 공개이므로 여기에 올린 모델·로그·영상도 공개됩니다.
import os
# 개인 사본에서 직접 지정할 경우 아래 한 줄의 주석을 풀어 사용하세요.
# os.environ['GH_TOKEN'] = '본인 토큰'
experiment.connect()
experiment.show_results()


In [ ]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


In [ ]:
# 05 · GitHub 데이터 다운로드·검증 / GPU와 정책 실행 확인
experiment.prepare_data()
experiment.check_runtime()


In [ ]:
# 06 · 노트북에 고정된 성능 근거와 입력 파일 식별자
import json, hashlib, copy, shutil
from pathlib import Path
from IPython.display import display, Video
EVIDENCE = json.loads('{"easy": {"run": "moveboxes_easy_lab_v1_benchmark", "checkpoint": "block_03.pt", "checkpoint_sha256": "0fbbef92aacc5cbe5642568ec5bd349979934e50f3d03d5120ccd5350720ef19", "score": 1.0, "n_episodes": 100, "sorted_boxes": 200.0, "total_boxes": 200, "seeds": [70000, 70001, 70002, 70003, 70004, 70005, 70006, 70007, 70008, 70009, 70010, 70011, 70012, 70013, 70014, 70015, 70016, 70017, 70018, 70019, 70020, 70021, 70022, 70023, 70024, 70025, 70026, 70027, 70028, 70029, 70030, 70031, 70032, 70033, 70034, 70035, 70036, 70037, 70038, 70039, 70040, 70041, 70042, 70043, 70044, 70045, 70046, 70047, 70048, 70049, 70050, 70051, 70052, 70053, 70054, 70055, 70056, 70057, 70058, 70059, 70060, 70061, 70062, 70063, 70064, 70065, 70066, 70067, 70068, 70069, 70070, 70071, 70072, 70073, 70074, 70075, 70076, 70077, 70078, 70079, 70080, 70081, 70082, 70083, 70084, 70085, 70086, 70087, 70088, 70089, 70090, 70091, 70092, 70093, 70094, 70095, 70096, 70097, 70098, 70099], "policy_config": {"model_config": {"state_dim": 54, "history": 16, "chunk_size": 16, "width": 128, "heads": 4, "layers": 2, "latent_dim": 16}, "temporal_decay": 0.25, "ensemble_window": 4, "gate_threshold": 0.65, "stage_threshold": 0.6, "act_horizon": 1, "num_inference_steps": 1}, "max_steps": 200, "repo_commit": "6048f33217f26ae39009a812f53c81171517f393", "metric_source": "easy_initial_audit/easy/metrics.json", "metric_sha256": "84bc97a78d7b795acc2c97f0ef32873593a09bf2413cc2ff6c8e220f9cc79474", "entries": [{"path": "easy/checkpoints/block_03.pt", "asset": "file-e8495370f5a21be0-0fbbef92aacc5cbe5642568ec5bd349979934e50f3d03d5120ccd5350720ef19.pt", "sha256": "0fbbef92aacc5cbe5642568ec5bd349979934e50f3d03d5120ccd5350720ef19", "bytes": 5132112}, {"path": "easy/checkpoints/policy_config.json", "asset": "file-db7efd25fba549cf-3a15086074d782696084a780fe871f817ad17cd34161199fac471b9ed83d19a8.json", "sha256": "3a15086074d782696084a780fe871f817ad17cd34161199fac471b9ed83d19a8", "bytes": 311}, {"path": "easy/collection/episode_100000.npz", "asset": "file-8ddaaffc3cb044c9-be2d5de918eab546299d3887fd2472ebf4e57ca00a2833cbaafc56588b55398f.npz", "sha256": "be2d5de918eab546299d3887fd2472ebf4e57ca00a2833cbaafc56588b55398f", "bytes": 19670}, {"path": "easy/collection/episode_100001.npz", "asset": "file-7cec35246798a9b1-fa482a31bacf2a63fe7c117c7a872834f9ed227d6dcf94d61ba24a95670ca74e.npz", "sha256": "fa482a31bacf2a63fe7c117c7a872834f9ed227d6dcf94d61ba24a95670ca74e", "bytes": 19858}, {"path": "easy/collection/episode_100002.npz", "asset": "file-8f9457e6b2261ab6-7393248b3bbb7a1c4e82f9ee01638b3ec701f0dcfd6e1f69809b69ce040c9a66.npz", "sha256": "7393248b3bbb7a1c4e82f9ee01638b3ec701f0dcfd6e1f69809b69ce040c9a66", "bytes": 24348}, {"path": "easy/collection/episode_100003.npz", "asset": "file-23fb1e4fc0fcef33-a03b549b0cb9ecdd9c6d23f2742c055e6b3262aba44775600f5ea06b4d0c326e.npz", "sha256": "a03b549b0cb9ecdd9c6d23f2742c055e6b3262aba44775600f5ea06b4d0c326e", "bytes": 19497}, {"path": "easy/collection/episode_100004.npz", "asset": "file-8e6ece8d86234334-32ffdcae4ade28177c2c45ce9af89956f1379618883b8d6b1aafc22a2798381f.npz", "sha256": "32ffdcae4ade28177c2c45ce9af89956f1379618883b8d6b1aafc22a2798381f", "bytes": 19720}, {"path": "easy/collection/episode_100005.npz", "asset": "file-0a1a95bb85f52a0f-34864fec8e2b466ab1ab90adde6503a5ba5368c4cda3f1298d252e2cc3ffba25.npz", "sha256": "34864fec8e2b466ab1ab90adde6503a5ba5368c4cda3f1298d252e2cc3ffba25", "bytes": 19448}, {"path": "easy/collection/episode_100006.npz", "asset": "file-34f2ed01bbe71677-2c9b8d6e5efd7599a2ed511da8b490e6f26d0862ca89d50226915fdfd8c2fdc2.npz", "sha256": "2c9b8d6e5efd7599a2ed511da8b490e6f26d0862ca89d50226915fdfd8c2fdc2", "bytes": 19496}, {"path": "easy/collection/episode_100007.npz", "asset": "file-e2b0f31ea14824e4-a5010f2b604505aea9aa065534f103819e91e96a1295d76f76c7df41c2534f8f.npz", "sha256": "a5010f2b604505aea9aa065534f103819e91e96a1295d76f76c7df41c2534f8f", "bytes": 16775}, {"path": "easy/collection/episode_100008.npz", "asset": "file-67b895037b1ee6f0-0c5629cb87bbfade38c1a627cb4dece9ef03fe60f5decb9c181e15784b5c6584.npz", "sha256": "0c5629cb87bbfade38c1a627cb4dece9ef03fe60f5decb9c181e15784b5c6584", "bytes": 19338}, {"path": "easy/collection/episode_100009.npz", "asset": "file-2c9e7ef785c45093-5a544ace1a0ba1a0b5a9bb7e7dba6d022de34ecd824bb8232d785ed49f790416.npz", "sha256": "5a544ace1a0ba1a0b5a9bb7e7dba6d022de34ecd824bb8232d785ed49f790416", "bytes": 19361}, {"path": "easy/collection/episode_100010.npz", "asset": "file-6efbfc541f1a2f37-7d03c91676f3f8e083f4578982a701bed38f6308e4318c48e8da6a6e03cdfdc4.npz", "sha256": "7d03c91676f3f8e083f4578982a701bed38f6308e4318c48e8da6a6e03cdfdc4", "bytes": 19575}, {"path": "easy/collection/episode_100011.npz", "asset": "file-eaad04948ef48d06-14f60f52d2417746f37ea766a9d40039017c6a06b6e8024e1115461e4aacc3ec.npz", "sha256": "14f60f52d2417746f37ea766a9d40039017c6a06b6e8024e1115461e4aacc3ec", "bytes": 19346}, {"path": "easy/collection/episode_100012.npz", "asset": "file-8b712bdb93f082f3-663a8d3a65927c9f6850aa3799b01b2bcb8ac7550fe816f964b3319cb5d20aa3.npz", "sha256": "663a8d3a65927c9f6850aa3799b01b2bcb8ac7550fe816f964b3319cb5d20aa3", "bytes": 24332}, {"path": "easy/collection/episode_100013.npz", "asset": "file-564fc968bedecdc7-c5f62e85b59c1ea1158bc9a483e19a1f03f401fbeba44792e779b0791235ffc3.npz", "sha256": "c5f62e85b59c1ea1158bc9a483e19a1f03f401fbeba44792e779b0791235ffc3", "bytes": 19997}, {"path": "easy/collection/episode_100014.npz", "asset": "file-80b2f971bf07591a-adffd961b21f78040aa66c445a996f7262d5e42e33230755ca0ec63c4d966243.npz", "sha256": "adffd961b21f78040aa66c445a996f7262d5e42e33230755ca0ec63c4d966243", "bytes": 19301}, {"path": "easy/collection/episode_100015.npz", "asset": "file-94c88191e3d2b80a-72f37503194d412e4debc4ceedb498bd4280dd878625147a031a8a0b8c033f62.npz", "sha256": "72f37503194d412e4debc4ceedb498bd4280dd878625147a031a8a0b8c033f62", "bytes": 16128}, {"path": "easy/collection/manifest.json", "asset": "file-59cefc5420701ac8-1489f172bf2afdb66d4673fc7f2f872a9a609a7deb64ff0ea3aa48fee88a7b68.json", "sha256": "1489f172bf2afdb66d4673fc7f2f872a9a609a7deb64ff0ea3aa48fee88a7b68", "bytes": 9281}, {"path": "easy/initial_model.pt", "asset": "file-a47915c7c41d5031-347341adff9e7244bc8a1a6634b741980dd10ab7353de036a60dea7fe756d336.pt", "sha256": "347341adff9e7244bc8a1a6634b741980dd10ab7353de036a60dea7fe756d336", "bytes": 15419147}, {"path": "easy/stage_train_job.json", "asset": "file-62274cf15e8302e8-331b86431e3eb157eaea6fc0b8cfc5cf415df4ecf56d9700ff41ebf27e359b71.json", "sha256": "331b86431e3eb157eaea6fc0b8cfc5cf415df4ecf56d9700ff41ebf27e359b71", "bytes": 1439}], "replay_stop": 6000}, "medium": {"run": "moveboxes_medium_lab_v1_benchmark", "checkpoint": "block_02.pt", "checkpoint_sha256": "f7c0d5edf5b2bd6b93918d10af43b9c4ee023befd3162139e391b371e1541730", "score": 0.40625, "n_episodes": 8, "sorted_boxes": 13.0, "total_boxes": 32, "seeds": [40000, 40001, 40002, 40003, 40004, 40005, 40006, 40007], "policy_config": {"model_config": {"state_dim": 72, "history": 16, "chunk_size": 16, "width": 128, "heads": 4, "layers": 2, "latent_dim": 16}, "temporal_decay": 0.25, "ensemble_window": 4, "gate_threshold": 0.65, "stage_threshold": 0.6, "act_horizon": 1, "num_inference_steps": 1}, "max_steps": 200, "repo_commit": "6048f33217f26ae39009a812f53c81171517f393", "metric_source": "medium_lab_audit/medium/test_metrics.json", "metric_sha256": "d9a251ffdb0e68a6a2d4576e868687911930f719001312d0392364720d59ca3e", "entries": [{"path": "medium/checkpoints/block_02.pt", "asset": "file-13ed57ec591572a3-f7c0d5edf5b2bd6b93918d10af43b9c4ee023befd3162139e391b371e1541730.pt", "sha256": "f7c0d5edf5b2bd6b93918d10af43b9c4ee023befd3162139e391b371e1541730", "bytes": 5639248}, {"path": "medium/checkpoints/policy_config.json", "asset": "file-925c47252244ffa9-7df9b69cacc292a02728204889c6e51b294aa906eca49c7df44b790c872a2c87.json", "sha256": "7df9b69cacc292a02728204889c6e51b294aa906eca49c7df44b790c872a2c87", "bytes": 311}, {"path": "medium/collection/episode_110000.npz", "asset": "file-1d8ba4ce46ce7057-6bca083788bcbabd27d3aeab37c2e8b6123f8499c6689c4bebcad1e4041c46a3.npz", "sha256": "6bca083788bcbabd27d3aeab37c2e8b6123f8499c6689c4bebcad1e4041c46a3", "bytes": 43841}, {"path": "medium/collection/episode_110001.npz", "asset": "file-a0b72e382de81972-3ee57f1aa953cd77521b6238b6bf5a7defdcb4aaf6edded7d1d0c429d1007a43.npz", "sha256": "3ee57f1aa953cd77521b6238b6bf5a7defdcb4aaf6edded7d1d0c429d1007a43", "bytes": 43772}, {"path": "medium/collection/episode_110003.npz", "asset": "file-34abc626af012e96-059edbc7ff4f9093ae8d1985e3b5109d3f78b7440cdf1571788916259b9d852b.npz", "sha256": "059edbc7ff4f9093ae8d1985e3b5109d3f78b7440cdf1571788916259b9d852b", "bytes": 49160}, {"path": "medium/collection/episode_110004.npz", "asset": "file-95eff3565f77894c-efe645c22402b9b65d4ed4349fdd8d38af07082e84f0c94024b2ae061b4ffe2b.npz", "sha256": "efe645c22402b9b65d4ed4349fdd8d38af07082e84f0c94024b2ae061b4ffe2b", "bytes": 41128}, {"path": "medium/collection/episode_110005.npz", "asset": "file-5d32d1043a696034-94109a135dcf19867cd53cb2cdd9efddb08f2cc6163050c02986549ac4c3464d.npz", "sha256": "94109a135dcf19867cd53cb2cdd9efddb08f2cc6163050c02986549ac4c3464d", "bytes": 41363}, {"path": "medium/collection/episode_110006.npz", "asset": "file-c06531d279c804d8-f6893518a7f908794c9956b25971b9bd68a3d5fe4d5ce66edc87ff039cbd83e4.npz", "sha256": "f6893518a7f908794c9956b25971b9bd68a3d5fe4d5ce66edc87ff039cbd83e4", "bytes": 43536}, {"path": "medium/collection/episode_110007.npz", "asset": "file-610673d7282efd45-c851e0c8846c3653521a54a6ed0c457be4c8fd8864930ac273c1b4bb22c29596.npz", "sha256": "c851e0c8846c3653521a54a6ed0c457be4c8fd8864930ac273c1b4bb22c29596", "bytes": 44232}, {"path": "medium/collection/episode_110008.npz", "asset": "file-68578e1cd28ec291-a9bdd78287dd798c84f2aabd38b39cde79173ffbecf4e7d40f981f8d5ac2c3a2.npz", "sha256": "a9bdd78287dd798c84f2aabd38b39cde79173ffbecf4e7d40f981f8d5ac2c3a2", "bytes": 48812}, {"path": "medium/collection/episode_110009.npz", "asset": "file-b9a4839815d55629-5d384b5bb37f06c56d7c51d17ec82f98b3555e081bbd0096a1d49a41382a66bf.npz", "sha256": "5d384b5bb37f06c56d7c51d17ec82f98b3555e081bbd0096a1d49a41382a66bf", "bytes": 48997}, {"path": "medium/collection/episode_110010.npz", "asset": "file-3218bed97360fce7-260c5d58fac577a895f5ada013b92d5bd13265997ffa62694ce4d74355fe7210.npz", "sha256": "260c5d58fac577a895f5ada013b92d5bd13265997ffa62694ce4d74355fe7210", "bytes": 49142}, {"path": "medium/collection/episode_110011.npz", "asset": "file-63762853243b01c2-5556314f8f793c150928782cc99a042a58a73efbb179d76104426c499a8bec33.npz", "sha256": "5556314f8f793c150928782cc99a042a58a73efbb179d76104426c499a8bec33", "bytes": 43540}, {"path": "medium/collection/episode_110012.npz", "asset": "file-8d4ac0fe227f541a-018626a2fcd44631a085b3c0a677f83fa7665fe7d065a05f30aa0057d7bbd035.npz", "sha256": "018626a2fcd44631a085b3c0a677f83fa7665fe7d065a05f30aa0057d7bbd035", "bytes": 36897}, {"path": "medium/collection/episode_110013.npz", "asset": "file-785726e02423f980-cd4e7550039d4505309800b7fca02f52cb64492a4c1dad70ae14042009b3178d.npz", "sha256": "cd4e7550039d4505309800b7fca02f52cb64492a4c1dad70ae14042009b3178d", "bytes": 43482}, {"path": "medium/collection/episode_110014.npz", "asset": "file-37941580b41faffb-d9dbf8fb64313dbed94301a076acfe70a16a182589a7a9b9e9fcf093890df4ee.npz", "sha256": "d9dbf8fb64313dbed94301a076acfe70a16a182589a7a9b9e9fcf093890df4ee", "bytes": 40002}, {"path": "medium/collection/episode_110015.npz", "asset": "file-689c3fe86e618ce0-a7f65410300b6bae6b5a256dacc510c8c966f0779beb5e0c2ba4cde3929bcb3b.npz", "sha256": "a7f65410300b6bae6b5a256dacc510c8c966f0779beb5e0c2ba4cde3929bcb3b", "bytes": 48878}, {"path": "medium/collection/episode_110016.npz", "asset": "file-3f8f966c62a583cb-684737148e13c98c2746c50414b192b83aa2ffb20f6d3b556d2474ea00a32100.npz", "sha256": "684737148e13c98c2746c50414b192b83aa2ffb20f6d3b556d2474ea00a32100", "bytes": 42338}, {"path": "medium/collection/manifest.json", "asset": "file-6b664448be020d9d-02244159c967f25160df5c70331f08e2f64b504a9a9b1bb217955aec6ed7a9f8.json", "sha256": "02244159c967f25160df5c70331f08e2f64b504a9a9b1bb217955aec6ed7a9f8", "bytes": 9563}, {"path": "medium/initial_model.pt", "asset": "file-b120fe6e1d74c4ec-fecd51d71cd7d9bdc4b7c58228d3350f8da429785ab9f28a58005a663481ad58.pt", "sha256": "fecd51d71cd7d9bdc4b7c58228d3350f8da429785ab9f28a58005a663481ad58", "bytes": 16940043}, {"path": "medium/stage_train_job.json", "asset": "file-f3a498bd596a9103-6161bc7ff36e1532f49fc131cbe6eeb25e4e729e2b84500694850848694e7180.json", "sha256": "6161bc7ff36e1532f49fc131cbe6eeb25e4e729e2b84500694850848694e7180", "bytes": 1454}], "replay_stop": 4000}, "hard": {"run": "moveboxes_hard_stage_v1_benchmark", "checkpoint": "best_val.pt", "checkpoint_sha256": "96d30fe99cdf6c684d8ad5a0e55fbc9b663d76b6ad02f2e6b331a1ea02da30bc", "score": 0.041666666666666664, "n_episodes": 8, "sorted_boxes": 2.0, "total_boxes": 48, "seeds": [48000, 48001, 48002, 48003, 48004, 48005, 48006, 48007], "policy_config": {"model_config": {"state_dim": 90, "history": 16, "chunk_size": 16, "width": 128, "heads": 4, "layers": 2, "latent_dim": 16}, "temporal_decay": 0.25, "ensemble_window": 4, "gate_threshold": 0.65, "stage_threshold": 0.6, "act_horizon": 1, "num_inference_steps": 1}, "max_steps": 200, "repo_commit": "6048f33217f26ae39009a812f53c81171517f393", "metric_source": "hard_audit/hard/test_metrics.json", "metric_sha256": "4dabcedc8e1398525841bc61dc16d7bd557fb5ee01c01d394b41a34719534883", "entries": [{"path": "hard/checkpoints/best_val.pt", "asset": "file-d1ba41539999b597-96d30fe99cdf6c684d8ad5a0e55fbc9b663d76b6ad02f2e6b331a1ea02da30bc.pt", "sha256": "96d30fe99cdf6c684d8ad5a0e55fbc9b663d76b6ad02f2e6b331a1ea02da30bc", "bytes": 6147472}, {"path": "hard/checkpoints/policy_config.json", "asset": "file-e09ac16bf24ff7ee-e889432140592a66c7ec30b2d9302f2d05a4912495434b64114159734cdc4f7f.json", "sha256": "e889432140592a66c7ec30b2d9302f2d05a4912495434b64114159734cdc4f7f", "bytes": 311}, {"path": "hard/collection/episode_160000.npz", "asset": "file-b99ba3d4bbf1b6ee-49e439fef160c7608652ee674509583ec336a8442e65645dada6a8872d9ffdca.npz", "sha256": "49e439fef160c7608652ee674509583ec336a8442e65645dada6a8872d9ffdca", "bytes": 77252}, {"path": "hard/collection/episode_160001.npz", "asset": "file-dcf6bb523ddc1212-75b5567bd4a30e93a4b3f570eeecc3a6a034297e3e1c23ae990ef80832cd7ed8.npz", "sha256": "75b5567bd4a30e93a4b3f570eeecc3a6a034297e3e1c23ae990ef80832cd7ed8", "bytes": 70804}, {"path": "hard/collection/episode_160002.npz", "asset": "file-b55e010a1cedfcde-ea2d6d7274b8a87a3d7fe3cc157cbd941e11918eafbdb376515e4889ca094b63.npz", "sha256": "ea2d6d7274b8a87a3d7fe3cc157cbd941e11918eafbdb376515e4889ca094b63", "bytes": 143573}, {"path": "hard/collection/episode_160003.npz", "asset": "file-3ae4ff1362399237-6283ff23f4d927de738b3f5ed040d224fff7f9c4b7bbcd22fb0d7215bc88733f.npz", "sha256": "6283ff23f4d927de738b3f5ed040d224fff7f9c4b7bbcd22fb0d7215bc88733f", "bytes": 76722}, {"path": "hard/collection/episode_160004.npz", "asset": "file-9c359d8409e6429b-4f9ced2d54f6cde000a16348a0f37c23acd01b2ca39c92708e658e0746323eca.npz", "sha256": "4f9ced2d54f6cde000a16348a0f37c23acd01b2ca39c92708e658e0746323eca", "bytes": 78867}, {"path": "hard/collection/episode_160005.npz", "asset": "file-278b9810dce7133a-9720ec9bc4fea3a5c89b6334eecf4e2b0d65131d516242f8210707462c61b654.npz", "sha256": "9720ec9bc4fea3a5c89b6334eecf4e2b0d65131d516242f8210707462c61b654", "bytes": 63641}, {"path": "hard/collection/episode_160008.npz", "asset": "file-5eaf09dbff139a34-2636cd70bec2e526c15deb9c02fc45662001db89d11562f9be55686ecc733976.npz", "sha256": "2636cd70bec2e526c15deb9c02fc45662001db89d11562f9be55686ecc733976", "bytes": 70092}, {"path": "hard/collection/episode_160009.npz", "asset": "file-8721e1b849e83aeb-e29765cf491870c42300e3c976643b4ef5f6d83bea358335b4edce4412c049b2.npz", "sha256": "e29765cf491870c42300e3c976643b4ef5f6d83bea358335b4edce4412c049b2", "bytes": 69719}, {"path": "hard/collection/episode_160010.npz", "asset": "file-64993698640b9d25-f83bbbff4987f7757ceddb158fb0e942150ff72f1858fa63e3f218814da6fc7c.npz", "sha256": "f83bbbff4987f7757ceddb158fb0e942150ff72f1858fa63e3f218814da6fc7c", "bytes": 69867}, {"path": "hard/collection/episode_160011.npz", "asset": "file-417463aaa5f2c762-256b7e106b9a13ef6968252382dd19c39bede0d615030094088feb37d0f6c87c.npz", "sha256": "256b7e106b9a13ef6968252382dd19c39bede0d615030094088feb37d0f6c87c", "bytes": 75917}, {"path": "hard/collection/episode_160012.npz", "asset": "file-766c2403df2107e9-54a0130c76139e1b1170b0deeaea7de7cdd9c81fb03b9efa5ef71b023d3ced4d.npz", "sha256": "54a0130c76139e1b1170b0deeaea7de7cdd9c81fb03b9efa5ef71b023d3ced4d", "bytes": 71068}, {"path": "hard/collection/episode_160013.npz", "asset": "file-106669de920c865a-d22867c7c37ae21488008d06e64cc590fd1d0438a66837f6923c7da8087ef0ea.npz", "sha256": "d22867c7c37ae21488008d06e64cc590fd1d0438a66837f6923c7da8087ef0ea", "bytes": 74838}, {"path": "hard/collection/episode_160014.npz", "asset": "file-8dca533561774b0b-4f3f343df09f0b060dff4392eee93ac65e92c2f495ae0d3717a8da6c24ffafaa.npz", "sha256": "4f3f343df09f0b060dff4392eee93ac65e92c2f495ae0d3717a8da6c24ffafaa", "bytes": 99541}, {"path": "hard/collection/episode_160015.npz", "asset": "file-494ced70d46a64af-bf83851fe8a9122e022624acc5b51d4178d03cde7d6dea588ad3199440a143da.npz", "sha256": "bf83851fe8a9122e022624acc5b51d4178d03cde7d6dea588ad3199440a143da", "bytes": 70452}, {"path": "hard/collection/episode_160016.npz", "asset": "file-5fec3f00c4ff6882-855dc9823ad3402cecc04834699cdcd1f669cb1c1045cad78b76b6f894b238bb.npz", "sha256": "855dc9823ad3402cecc04834699cdcd1f669cb1c1045cad78b76b6f894b238bb", "bytes": 117593}, {"path": "hard/collection/episode_160018.npz", "asset": "file-b8299879606e0251-330d9b2a6a81a89014f411b3a38f14ce6071637e2b137cc525433be7d7536b7d.npz", "sha256": "330d9b2a6a81a89014f411b3a38f14ce6071637e2b137cc525433be7d7536b7d", "bytes": 87691}, {"path": "hard/collection/episode_160019.npz", "asset": "file-e2a06bddedf81e0c-8ce0b605cb16101488cd85581b8bba12ca558c05c85f8aa10ca0010c7e911f60.npz", "sha256": "8ce0b605cb16101488cd85581b8bba12ca558c05c85f8aa10ca0010c7e911f60", "bytes": 73927}, {"path": "hard/collection/episode_160020.npz", "asset": "file-1dc82429c2708ecd-13db5574875585044e44a8da5dada38f9d3eee273bd4e50719245345731bacb2.npz", "sha256": "13db5574875585044e44a8da5dada38f9d3eee273bd4e50719245345731bacb2", "bytes": 75935}, {"path": "hard/collection/episode_160022.npz", "asset": "file-0581d3c37beea618-21babd1af1b14fc6a6507ea66bb78e010a7ea220a6d6b16a6740c9899e04a12a.npz", "sha256": "21babd1af1b14fc6a6507ea66bb78e010a7ea220a6d6b16a6740c9899e04a12a", "bytes": 74757}, {"path": "hard/collection/episode_160023.npz", "asset": "file-df4b296ae2a5124c-6b8c40dc1a696403088f19536aa089fc6d3884e9f2e82e60d5a7e05e9e041110.npz", "sha256": "6b8c40dc1a696403088f19536aa089fc6d3884e9f2e82e60d5a7e05e9e041110", "bytes": 74596}, {"path": "hard/collection/episode_160024.npz", "asset": "file-f0f3be533eb9ce62-9443200d1997f26f8ca5ebeb5fbfb2ad424f7543f6a0c4d0834c1819c784ae6e.npz", "sha256": "9443200d1997f26f8ca5ebeb5fbfb2ad424f7543f6a0c4d0834c1819c784ae6e", "bytes": 76347}, {"path": "hard/collection/episode_160026.npz", "asset": "file-d580ae82c1287684-7e646d221f64114605a4e61a447846ef28cd60a9f3c03d8f3734fc041d0ead5d.npz", "sha256": "7e646d221f64114605a4e61a447846ef28cd60a9f3c03d8f3734fc041d0ead5d", "bytes": 67471}, {"path": "hard/collection/episode_160027.npz", "asset": "file-52fc18985710293d-836849dc8930d4c1472babc5e2101f8e932d8bca55b37540a52da06354ab496a.npz", "sha256": "836849dc8930d4c1472babc5e2101f8e932d8bca55b37540a52da06354ab496a", "bytes": 91358}, {"path": "hard/collection/episode_160028.npz", "asset": "file-23238d6bd8c7d223-7edbd46fa7ed3595d6d4491d52ab85c3bffd4d67fd6bbf6ab0c8d223a55ebb47.npz", "sha256": "7edbd46fa7ed3595d6d4491d52ab85c3bffd4d67fd6bbf6ab0c8d223a55ebb47", "bytes": 97709}, {"path": "hard/collection/manifest.json", "asset": "file-54d6111d16ece3b7-4622313338b1a3d18be6f051659e576edc155ac93c2265cb9715a297e9c1afbd.json", "sha256": "4622313338b1a3d18be6f051659e576edc155ac93c2265cb9715a297e9c1afbd", "bytes": 15207}, {"path": "hard/stage_train_job.json", "asset": "file-147f09bc1567e045-8a4ea90cdec8121a70757a5f2d7e9f97898b7812634687c811fd92a886a441f5.json", "sha256": "8a4ea90cdec8121a70757a5f2d7e9f97898b7812634687c811fd92a886a441f5", "bytes": 1393}], "replay_stop": 30000}}')
assert set(TRAIN_LEVELS) <= set(EVIDENCE), 'TRAIN_LEVELS의 난이도를 확인하세요.'
assert EVAL_PROTOCOL in ('historical', 'official_default')
for level, record in EVIDENCE.items():
    print(level, f"{record['score']:.4%}", record['n_episodes'], 'episodes', record['checkpoint_sha256'])

In [ ]:
# 07 · 최고 가중치 / 당시 job / 당시 초기값 / 복구 시연을 정확한 해시로 복원
from github_store import GitHubStore, safe_target
from marso_experiment import digest, read_json, save_json
from stage_anchor_continue import package

ROOT_RUN = Path(experiment.run_dir)
INPUT_ROOT = ROOT_RUN/'historical_inputs'
ORIGINAL_JOBS = {}

def fetch_exact(store, entry, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        if target.stat().st_size != entry['bytes'] or digest(target) != entry['sha256']:
            raise ValueError(f'로컬 파일이 고정된 과거 파일과 다릅니다: {target}')
        return target
    asset = store.assets.get(entry['asset'])
    if asset is None or asset['size'] != entry['bytes']:
        raise FileNotFoundError(f"정확한 Release asset이 없습니다: {entry['asset']}")
    pending = target.with_name(target.name+'.part')
    store.download(asset, pending)
    if pending.stat().st_size != entry['bytes'] or digest(pending) != entry['sha256']:
        raise ValueError(f'다운로드 SHA-256 불일치: {target}')
    pending.replace(target)
    return target

import torch
for level, record in EVIDENCE.items():
    store = GitHubStore(CFG['github_repository'], 'run-'+record['run'])
    if not store.load(create=False):
        raise FileNotFoundError(record['run'])
    for entry in record['entries']:
        fetch_exact(store, entry, safe_target(INPUT_ROOT, entry['path']))
    source = INPUT_ROOT/level/'checkpoints'/record['checkpoint']
    anchor = ROOT_RUN/level/'anchor.pt'
    anchor.parent.mkdir(parents=True, exist_ok=True)
    if anchor.exists() and digest(anchor) != record['checkpoint_sha256']:
        raise ValueError(f'{level}: 이전 anchor와 충돌합니다. 새 run_name을 사용하세요.')
    if not anchor.exists():
        shutil.copy2(source, anchor)
    state = torch.load(anchor, map_location='cpu', weights_only=True)
    assert state['format'] == 'moveboxes-stage-act-v1'
    assert state['model_config'] == record['policy_config']['model_config']
    job = read_json(INPUT_ROOT/level/'stage_train_job.json')
    assert job['model_config'] == state['model_config']
    ORIGINAL_JOBS[level] = job
    print(level, 'checkpoint step =', state['step'])
    print('당시 train_config:', json.dumps(job['train_config'], indent=2))
    print('당시 warm_start:', job.get('warm_start', '없음'))
    save_json(ROOT_RUN/level/'historical_evidence.json', record)
    experiment.sync_level(level)

BASELINE = package(experiment, folder_name='historical_best_candidate')
save_json(ROOT_RUN/'historical_evidence.json', EVIDENCE)
print('검증된 과거 최고 모델 패키지:', BASELINE)

## 3. 최고 가중치 재평가

아래는 모델을 새로 학습하지 않고 공식 `eval.py`를 실행합니다. `historical`은 표의 seed와 episode 수를 사용하고,
`official_default`는 공식 default 설정을 사용합니다. 실행 코드 버전 차이 때문에 과거 점수와 달라질 수 있습니다.
공식 evaluator의 반환 metrics를 기록하며, 인쇄된 반올림 점수만 읽어 비교하지 않습니다.
기억을 가진 정책은 episode/영상 시작 시 초기화합니다. 영상은 평가 결과와 함께 표시합니다.

In [ ]:
# 08 · 공식 평가 함수 + Easy
import os, sys, subprocess

# 공식 eval.py의 rollout 계산은 그대로 사용하고 반환값만 JSON으로 보존합니다.
EVAL_RUNNER = ROOT_RUN/'official_capture.py'
EVAL_RUNNER.write_text("""import json, runpy, sys
from pathlib import Path
import warehouse_sort.utils as utils
job = json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
original_metrics = utils.rollout_metrics
original_video = utils.record_eval_video
def capture(*args, **kwargs):
    metrics = original_metrics(*args, **kwargs)
    Path(job['metrics']).write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    return metrics
def video(*args, **kwargs):
    agent = args[3] if len(args) > 3 else kwargs['agent']
    if hasattr(agent, 'reset'):
        agent.reset()
    return original_video(*args, **kwargs)
utils.rollout_metrics = capture
utils.record_eval_video = video
sys.argv = job['command']
runpy.run_path(job['command'][0], run_name='__main__')
""", encoding='utf-8')

def evaluate_candidate(candidate, level, label):
    candidate = Path(candidate)
    folder = ROOT_RUN/level/'official_evaluations'/label/EVAL_PROTOCOL
    folder.mkdir(parents=True, exist_ok=True)
    record = EVIDENCE[level]
    if EVAL_PROTOCOL == 'historical':
        config = folder/'eval_config.yaml'
        config.write_text('eval:\n  n_episodes: '+str(record['n_episodes'])+
                          '\n  seeds: '+json.dumps(record['seeds'])+'\n', encoding='utf-8')
    else:
        config = Path(CFG['repo_dir'])/'conf/eval/default.yaml'
    checkpoint = candidate/'checkpoints'/level/'model.pt'
    metrics_file = folder/'metrics.json'
    command = [str(Path(CFG['repo_dir'])/'eval.py'), 'difficulty='+level,
        'obs_mode=state', 'num_envs=1', 'policy=stage_policy:load_policy',
        'checkpoint='+str(checkpoint), 'eval_config='+str(config),
        'max_episode_steps=200', 'hydra.run.dir='+str(folder)]
    identity = dict(checkpoint_sha256=digest(checkpoint),
        policy_config=read_json(checkpoint.parent/'policy_config.json'),
        code_commit=CFG['project_commit'], evaluator_sha256=digest(EVAL_RUNNER),
        eval_config_sha256=digest(config), max_steps=200, protocol=EVAL_PROTOCOL)
    done = read_json(folder/'complete.json', {})
    if done == identity and metrics_file.is_file():
        metrics = read_json(metrics_file)
        print(level, '완료 평가 재사용:', metrics['sort_accuracy'])
    else:
        metrics_file.unlink(missing_ok=True)
        job = folder/'capture_job.json'
        save_json(job, dict(command=command, metrics=str(metrics_file)))
        env = dict(os.environ)
        # 복원 package의 정책 코드가 simulator 작업 폴더의 코드보다 먼저 로드됩니다.
        env['PYTHONPATH'] = os.pathsep.join([str(candidate), CFG['repo_dir'], env.get('PYTHONPATH','')])
        log = folder/'official_eval.log'
        with log.open('w', encoding='utf-8') as handle:
            process = subprocess.Popen([sys.executable, str(EVAL_RUNNER), str(job)],
                cwd=CFG['repo_dir'], env=env, stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1)
            try:
                for line in process.stdout:
                    handle.write(line); handle.flush(); print(line, end='', flush=True)
                code = process.wait()
            finally:
                if process.poll() is None:
                    process.terminate()
                    try:
                        process.wait(timeout=5)
                    except subprocess.TimeoutExpired:
                        process.kill(); process.wait()
                process.stdout.close()
        if code or not metrics_file.is_file():
            raise RuntimeError(f'공식 평가 실패: {log}')
        metrics = read_json(metrics_file)
        save_json(folder/'complete.json', identity)
    print(level, EVAL_PROTOCOL, 'SORT ACCURACY:', f"{metrics['sort_accuracy']:.4%}")
    for video in sorted((folder/'videos').rglob('*.mp4'))[:1]:
        display(Video(str(video), embed=True, width=900))
    try:
        experiment.sync_level(level)
    except Exception as error:
        print('평가는 로컬 저장 완료, GitHub 백업 재시도 필요:', type(error).__name__)
    return metrics

BASELINE_RESULTS = {}
BASELINE_RESULTS['easy'] = evaluate_candidate(BASELINE, 'easy', 'historical_best')

In [ ]:
# 09 · MEDIUM 최고 가중치 재평가
BASELINE_RESULTS['medium'] = evaluate_candidate(BASELINE, 'medium', 'historical_best')

In [ ]:
# 10 · HARD 최고 가중치 재평가
BASELINE_RESULTS['hard'] = evaluate_candidate(BASELINE, 'hard', 'historical_best')

## 4. 당시 학습 루트 재실행 — 선택한 난이도만

01의 `TRAIN_LEVELS=[]`이면 아래 셀은 학습을 건너뜁니다. 예: `['medium']`이면 Medium만 실행합니다.
GPU 학습을 실제로 돌리고 싶을 때 설정을 변경합니다. 이것은 **당시 recipe의 재학습**이며 최고 모델에서 추가 fine-tuning하는 경로가 아닙니다.

- Easy·Medium: 당시 `initial_model.pt`와 시연을 복사한 새 run에서 시작합니다. 당시 total_iters=20,000을 유지하고 Easy 6,000 / Medium 4,000에서 멈춥니다.
- Hard: 당시 training job과 성공 시연으로 30,000까지 학습하고 새 `best_val.pt`를 따로 평가합니다. 과거 best_val과 동일해진다고 가정하지 않습니다.
- 과거 lab은 2,000회마다 평균화 window 1/4를 비교했습니다. 여기서는 **과거 최고에 사용된 window 4와 저장 step을 고정**해서 재학습을 재현합니다. 새 자동 탐색은 수행하지 않습니다.
- 저장된 scheduler 예산·batch·표집·warmup은 그대로 사용합니다. 새 학습의 source hash와 입력 hash를 함께 기록합니다.
- `latest.pt`의 optimizer/scaler/표집 RNG/torch RNG/CUDA RNG로 중단 재개합니다. 원격 복구는 마지막으로 업로드가 완료된 checkpoint 기준입니다.

In [ ]:
# 11 · 당시 입력과 설정으로 새 run 준비 / 재개
from stage_experiment import StageExperiment
TRAIN_RUNS = {}

def prepare_replay(level):
    record = EVIDENCE[level]
    original = copy.deepcopy(ORIGINAL_JOBS[level])
    cfg = copy.deepcopy(experiment.cfg)
    cfg['run_name'] = CFG['run_name']+'_'+level+'_recipe_replay_v1'
    cfg.update(original['train_config'])
    cfg['total_iters'] = dict(experiment.cfg['total_iters'])
    cfg['total_iters'][level] = original['train_config']['total_iters']
    child = StageExperiment(cfg, dict(experiment.sources))
    child.connect()
    child._stage_helpers()
    target = Path(child.run_dir)/level
    target.mkdir(parents=True, exist_ok=True)
    for entry in record['entries']:
        if '/collection/' not in entry['path'] and not entry['path'].endswith('/initial_model.pt'):
            continue
        source = safe_target(INPUT_ROOT, entry['path'])
        dest = safe_target(child.run_dir, entry['path'])
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() and digest(dest) != entry['sha256']:
            raise ValueError(f'입력이 달라졌습니다: {dest}')
        if not dest.exists():
            shutil.copy2(source, dest)
    job = copy.deepcopy(original)
    job['folder'] = str(target)
    job['data'] = str(child.data/level/'trajectory.state.pd_ee_delta_pos.physx_cuda.h5')
    job['recovery_manifest'] = str(target/'collection/manifest.json')
    if original.get('warm_start'):
        initial = target/'initial_model.pt'
        if not initial.is_file():
            raise FileNotFoundError('당시 초기 가중치가 없습니다. 다른 모델로 대체하지 않습니다.')
        job['warm_start'] = str(initial)
    elif (target/'initial_model.pt').exists():
        raise ValueError('원래 warm_start가 없는 학습에 초기 모델이 섞였습니다.')
    names = ('act_v2_model.py','act_v2_data.py','stage_schema.py','stage_labels.py',
             'stage_model.py','stage_data.py','stage_train.py')
    code_hash = hashlib.sha256(''.join(child.sources[n] for n in names).encode()).hexdigest()
    job['source_sha256'] = code_hash
    job['stop_at'] = record['replay_stop']
    origin = dict(historical_job=original, replay_job=job,
        historical_source_sha256=original['source_sha256'], current_source_sha256=code_hash,
        same_training_source=code_hash == original['source_sha256'],
        input_hashes={e['path']:e['sha256'] for e in record['entries']
                      if '/collection/' in e['path'] or e['path'].endswith('/initial_model.pt')},
        data_sha256=digest(job['data']), project_commit=CFG['project_commit'])
    old = read_json(target/'replay_origin.json')
    if old and old != origin:
        raise ValueError('이 run의 재학습 설정/코드/데이터가 달라졌습니다. 새 run_name을 사용하세요.')
    save_json(target/'replay_origin.json', origin)
    save_json(target/'stage_train_job.json', job)
    if not origin['same_training_source']:
        print(level, '학습 코드 hash가 당시 기록과 다릅니다. 새 코드의 recipe 재실행으로 기록합니다.')
    print(level, 'scheduler total =', job['train_config']['total_iters'], 'stop_at =', job['stop_at'])
    return child

for level in TRAIN_LEVELS:
    TRAIN_RUNS[level] = prepare_replay(level)
if not TRAIN_LEVELS:
    print('복원/평가 모드입니다. 재학습은 TRAIN_LEVELS에 난이도를 넣으면 실행됩니다.')

RETRAINED = {}
def replay_and_evaluate(level):
    if level not in TRAIN_LEVELS:
        print(level, '재학습 생략')
        return
    child = TRAIN_RUNS[level]
    folder = Path(child.run_dir)/level
    job_path = folder/'stage_train_job.json'
    with child.persist_operation(level):
        child.run([sys.executable, str(child.repo/'stage_train.py'), str(job_path)],
                  cwd=child.repo, log=folder/'recipe_replay_train.log')
    checkpoint = folder/'checkpoints'/('best_val.pt' if level == 'hard' else 'latest.pt')
    state = torch.load(checkpoint, map_location='cpu', weights_only=True)
    if level != 'hard':
        assert state['step'] == EVIDENCE[level]['replay_stop']
    # 추론용 파일에는 가중치만 저장하고 재개용 latest.pt는 학습 run에 보존합니다.
    frozen = folder/'checkpoints'/'recipe_inference.pt'
    torch.save({k:state[k] for k in ('format','model_config','model','step')}, frozen)
    candidate = package(experiment, checkpoint_overrides={level:frozen},
                        folder_name='recipe_replay_'+level+'_candidate')
    # 공통 packager의 기본 라벨을 실제 재학습 경로에 맞게 기록합니다.
    manifest = read_json(candidate/'manifest.json')
    manifest['levels'][level]['selection'] = 'historical_recipe_retraining'
    save_json(candidate/'manifest.json', manifest)
    result = evaluate_candidate(candidate, level, 'recipe_replay')
    RETRAINED[level] = dict(checkpoint=str(frozen), step=state['step'], metrics=result,
                            historical_score=EVIDENCE[level]['score'])
    save_json(folder/'recipe_result.json', RETRAINED[level])
    child.sync_level(level)
    print(level, '과거:', EVIDENCE[level]['score'], '새 평가:', result['sort_accuracy'])

In [ ]:
# 12 · EASY 당시 루트 재학습 + 공식 평가
replay_and_evaluate('easy')

In [ ]:
# 13 · MEDIUM 당시 루트 재학습 + 공식 평가
replay_and_evaluate('medium')

In [ ]:
# 14 · HARD 당시 루트 재학습 + 공식 평가
replay_and_evaluate('hard')

In [ ]:
# 15 · 과거 최고 모델 하나의 ZIP + 재학습 모델은 별도 ZIP
save_json(BASELINE/'historical_evidence.json', EVIDENCE)
save_json(BASELINE/'reevaluation_results.json', BASELINE_RESULTS)
archive = shutil.make_archive(str(ROOT_RUN/'moveboxes_historical_best'), 'zip', root_dir=BASELINE)
print('과거 최고 모델 ZIP:', archive)
print('새 평가 결과:', json.dumps(BASELINE_RESULTS, indent=2))
for level in RETRAINED:
    candidate = ROOT_RUN/('recipe_replay_'+level+'_candidate')
    save_json(candidate/'recipe_result.json', RETRAINED[level])
    output = shutil.make_archive(str(ROOT_RUN/('moveboxes_recipe_replay_'+level)), 'zip', root_dir=candidate)
    print('재학습 별도 ZIP:', output)
try:
    experiment.sync_common()
except Exception as error:
    print('ZIP은 로컬 저장 완료, 원격 백업 재시도 필요:', type(error).__name__)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    from IPython.display import FileLink
    display(FileLink(archive))

## 근거 파일과 해석

이 노트북의 `EVIDENCE` 셀에는 성능 기록의 상대 경로·SHA-256, 최고 checkpoint SHA-256,
당시 모델과 학습 입력의 Release asset 이름·크기·SHA-256, 정책 설정과 전체 평가 seed를 포함했습니다.
원래 PC의 경로가 없어도 Release에서 같은 입력을 받습니다. 상단 요약은 네트워크 없이 읽을 수 있습니다.

| 기록 | 로컬 근거 |
|---|---|
| Easy 최종 100회 / 학습 이력 | easy_initial_audit/easy/metrics.json, lab_history.json, lab_best.json |
| Medium 테스트 8회 / 개발 이력 | medium_lab_audit/medium/test_metrics.json, lab_history.json, stage_train_job.json |
| Hard 테스트 8회 / 완료 step / validation | hard_audit/hard/test_metrics.json, train_status.json, training_progress.json |
| 모델·학습 입력 식별 | 각 audit 폴더의 snapshot.json |
| 이후 v2.2 / Hard 전이 | performance_audit_current 아래 해당 run의 test_metrics.json, lab_best.json |

**검증 범위:** 생성 시 JSON/노트북 구조와 Python 셀 문법, 기록된 성능·모델 해시·평가 조건의 일치 여부를 확인했습니다.
이 노트북 전체의 Colab 설치·Release 다운로드·GPU 학습·시뮬레이션은 이번 정리 과정에서 실행하지 않았습니다.